In [1]:
# input
import pandas as pd
import argparse
import numpy as np

In [2]:
def get_peaks(data,bin_size_kb=2, comp_flanksize_kb=50, fold_thresh=5):
    scaff_l = list(set(data.Scaffold))
    peak = []
    nopeak = []
    
    for scaff in scaff_l:
        onekbfile = data.loc[data.Scaffold==scaff].reset_index()
        
        for i, bin in onekbfile.iterrows():
            #print(i)
            #print(scaff)
            #print(bin.Start_bp)
            #print(onekbfile)
            #print(onekbfile.iloc[i:i+bin_size_kb-1])
            peak_mean_rho = onekbfile.iloc[i:i+bin_size_kb-1].Rho_kb.mean()
            #print(peak_mean_rho)

            if i == onekbfile.index.max():
                peak_end = onekbfile.iloc[i,].End_bp
            else:
                try:
                    peak_end = onekbfile.iloc[i+1,].End_bp
                except IndexError:
                    #print(onekbfile.index.max())
                    #print(i)
                    #print(bin)
                    peak_end = onekbfile.iloc[i,].End_bp
                    raise IndexError
                    
                    
            flank_start = i-50
            if flank_start<0: # make sure we dont overshoot the scaffold boundary
                flank_start=0
    
            flank_stop = i+bin_size_kb-1+50
            if flank_stop>onekbfile.index.max(): # make sure we dont overshoot the scaffold boundary
                flank_stop=onekbfile.index.max()

            flank_mean_rho = onekbfile.iloc[flank_start:flank_stop].Rho_kb.mean()
            #print(scaff)
            #print(bin.Start_bp)
            #print(peak_end)
            #print(peak_mean_rho)
            #print(flank_mean_rho)
            #print(peak_mean_rho/flank_mean_rho)
            fold_diff = (peak_mean_rho/flank_mean_rho)
            if fold_diff>fold_thresh:
                #print("PEAK")
                peak.append([scaff, bin.Start_bp, peak_end, True, fold_diff, peak_mean_rho, flank_mean_rho])
            else:
                nopeak.append([scaff, bin.Start_bp, peak_end, True, fold_diff, peak_mean_rho, flank_mean_rho])
        #    else: 
        #        peak.append('False')
    return peak, nopeak
        
    

In [3]:
data = pd.read_csv("../data/concat_Csec_geq1000_rmind_hardfilt_exhet_biall_dp_qfilt_mac2_maxmiss06_phimp_LDhat_bpen1_statres_filtMQ70depth2stdev.txt_w1kb", sep='\t')

In [4]:
p, pn = get_peaks(data=data)

In [5]:
peaks = pd.DataFrame(p)

In [6]:
nopeaks = pd.DataFrame(pn)

In [7]:
peaks.columns = ["scaffold", "bin_Start_bp", "bin_Stop_bp", "peak", "fold_diff", "peak_mean_rho", "flank_mean_rho"]
nopeaks.columns = ["scaffold", "bin_Start_bp", "bin_Stop_bp", "peak", "fold_diff", "peak_mean_rho", "flank_mean_rho"]

In [8]:
peaks.bin_Start_bp = peaks.bin_Start_bp.astype(int).astype(str)
peaks.bin_Stop_bp = peaks.bin_Stop_bp.astype(int).astype(str)

In [9]:
nopeaks.bin_Start_bp = nopeaks.bin_Start_bp.astype(int).astype(str)
nopeaks.bin_Stop_bp = nopeaks.bin_Stop_bp.astype(int).astype(str)

In [10]:
peaks.to_csv('../data/20231204_Csec_5fold_peaks.bed', sep='\t', header=None, index=None)

In [15]:
nopeaks.to_csv('../data/20231205_Csec_5fold_nopeaks_all.tsv', sep='\t', header=True, index=None)

In [11]:
take_idx = np.random.choice( a= range(nopeaks.index.max()), size=peaks.shape[0], replace=False )

In [12]:
nopeaks_ss = nopeaks.iloc[take_idx]

In [175]:
nopeaks_ss[["scaffold", "bin_Start_bp",	"bin_Stop_bp"]].to_csv('../data/20231205_Csec_5fold_nopeaks.bed', sep='\t', header=None, index=None)

In [236]:
peaks.shape

(40894, 7)

# make subset for testing

In [179]:
peaks.sort_values(by='fold_diff', ascending=False)

,scaffold,bin_Start_bp,bin_Stop_bp,peak,fold_diff,peak_mean_rho,flank_mean_rho
460,NEVH01014529.1,2000,3000,True,2952.507329,11.95950,0.004051
35286,NEVH01008345.1,8000,9000,True,2395.215738,2.18982,0.000914
14751,NEVH01023493.1,1000,2000,True,345.645552,3.54024,0.010242
19432,NEVH01010544.1,1000,2000,True,278.568459,8.57651,0.030788
24332,NEVH01015377.1,2000,3000,True,197.058996,1.58183,0.008027
...,...,...,...,...,...,...,...
31945,NEVH01004401.1,1014000,1016000,True,5.000310,16.80200,3.360192
5241,NEVH01006739.1,128000,130000,True,5.000204,17.61630,3.523117
23464,NEVH01026110.1,393000,395000,True,5.000201,23.96550,4.792908
5654,NEVH01002152.1,341000,343000,True,5.000108,32.76020,6.551898


In [184]:
peaks.flank_mean_rho.mean()

3.84582936180377

In [237]:
peaks_ss = peaks.loc[peaks.flank_mean_rho > 3.84582936180377].sort_values(by='fold_diff', ascending=False)

In [238]:
peaks_ss

,scaffold,bin_Start_bp,bin_Stop_bp,peak,fold_diff,peak_mean_rho,flank_mean_rho
35540,NEVH01009080.1,1613000,1615000,True,25.100482,182.4660,7.269422
33825,NEVH01026396.1,69000,71000,True,24.235714,132.0590,5.448942
15629,NEVH01024944.1,1113000,1115000,True,21.443277,84.8822,3.958453
13906,NEVH01009768.1,2173000,2175000,True,21.411040,148.0720,6.915685
37170,NEVH01021187.1,335000,337000,True,21.299898,97.7582,4.589609
...,...,...,...,...,...,...,...
34716,NEVH01009372.1,869000,871000,True,5.000585,50.8930,10.177410
4493,NEVH01018385.1,838000,840000,True,5.000467,45.5603,9.111208
23464,NEVH01026110.1,393000,395000,True,5.000201,23.9655,4.792908
5654,NEVH01002152.1,341000,343000,True,5.000108,32.7602,6.551898


In [239]:
take_idx = np.random.choice( a= range(nopeaks.index.max()), size=18473, replace=False )

In [240]:
nopeaks_ss2 = nopeaks.iloc[take_idx]

In [241]:
peaks_ss.to_csv('../data/20231204_Csec_5fold_peaks_18kss.bed', sep='\t', header=None, index=None)
nopeaks_ss2[["scaffold", "bin_Start_bp",	"bin_Stop_bp"]].to_csv('../data/20231205_Csec_5fold_nopeaks_18kss.bed', sep='\t', header=None, index=None)

In [242]:
%%bash

db=../data/Csec.fa
bed=../data/20231204_Csec_5fold_peaks_18kss.bed
out=../data/20231204_Csec_hs_5fold_18kss.fa

bedtools getfasta -fi $db -bed $bed -fo $out


db=../data/Csec.fa
bed=../data/20231205_Csec_5fold_nopeaks_18kss.bed
out=../data/20231204_Csec_hs_5fold_background_18kss.fa

bedtools getfasta -fi $db -bed $bed -fo $out


Feature (NEVH01018006.1:3000-4000) beyond the length of NEVH01018006.1 size (3609 bp).  Skipping.
Feature (NEVH01000595.1:2118000-2119000) beyond the length of NEVH01000595.1 size (2118713 bp).  Skipping.
Feature (NEVH01010480.1:2306000-2307000) beyond the length of NEVH01010480.1 size (2306319 bp).  Skipping.
Feature (NEVH01017307.1:1000-2000) beyond the length of NEVH01017307.1 size (1319 bp).  Skipping.
Feature (NEVH01014373.1:254000-256000) beyond the length of NEVH01014373.1 size (255944 bp).  Skipping.
Feature (NEVH01004982.1:58000-59000) beyond the length of NEVH01004982.1 size (58611 bp).  Skipping.
Feature (NEVH01014373.1:255000-256000) beyond the length of NEVH01014373.1 size (255944 bp).  Skipping.
Feature (NEVH01009081.1:266000-268000) beyond the length of NEVH01009081.1 size (267562 bp).  Skipping.
Feature (NEVH01013552.1:1655000-1657000) beyond the length of NEVH01013552.1 size (1656948 bp).  Skipping.
Feature (NEVH01009081.1:267000-268000) beyond the length of NEVH010090

In [198]:
%%bash

db=../data/Csec.fa
bed=../data/20231204_Csec_5fold_peaks_4kss.bed
out=../data/20231204_Csec_hs_5fold_4kss.fa

bedtools getfasta -fi $db -bed $bed -fo $out


db=../data/Csec.fa
bed=../data/20231205_Csec_5fold_nopeaks_4kss.bed
out=../data/20231204_Csec_hs_5fold_background_4kss.fa

bedtools getfasta -fi $db -bed $bed -fo $out


Feature (NEVH01018006.1:3000-4000) beyond the length of NEVH01018006.1 size (3609 bp).  Skipping.
Feature (NEVH01000595.1:2118000-2119000) beyond the length of NEVH01000595.1 size (2118713 bp).  Skipping.
Feature (NEVH01010480.1:2306000-2307000) beyond the length of NEVH01010480.1 size (2306319 bp).  Skipping.
Feature (NEVH01017307.1:1000-2000) beyond the length of NEVH01017307.1 size (1319 bp).  Skipping.
Feature (NEVH01014373.1:254000-256000) beyond the length of NEVH01014373.1 size (255944 bp).  Skipping.
Feature (NEVH01004982.1:58000-59000) beyond the length of NEVH01004982.1 size (58611 bp).  Skipping.
Feature (NEVH01014373.1:255000-256000) beyond the length of NEVH01014373.1 size (255944 bp).  Skipping.
Feature (NEVH01009081.1:266000-268000) beyond the length of NEVH01009081.1 size (267562 bp).  Skipping.
Feature (NEVH01013552.1:1655000-1657000) beyond the length of NEVH01013552.1 size (1656948 bp).  Skipping.
Feature (NEVH01009081.1:267000-268000) beyond the length of NEVH010090

In [217]:
seqs = pd.read_csv("../data/streme_out_v2/sequences.tsv", sep='\t')
seqs = seqs.dropna(subset='motif_ALT_ID')
seqs['idnum'] = [ int(i.split('-')[1]) for i in seqs.motif_ALT_ID]
seqs.loc[seqs.idnum<13][['motif_ID', 'motif_P-value']].drop_duplicates()